# 03b — Video Mode 1: First/Last-Frame → Video (Wan2.1-FLF2V-14B, quality path)

Replaces the ComfyUI Wan FLF2V workflow. Pure diffusers: `WanImageToVideoPipeline` with
`image=` (start) + `last_image=` (end). This is the **higher-quality** Mode 1 option vs 03a's LTX:
Wan FLF2V gives sharper 720p frames and better identity retention, at the cost of a much bigger
model (~28 GB) and slower sampling (~50 steps). Use it for hero shots; use 03a (LTX) for the
cheap/fast iteration loop and long agentic chains.

**When to pick this over 03a:** you have an A100 and want the best two-frame interpolation.
**When to pick 03a instead:** multi-keyframe (start+mid+end), fast iteration, or <30 GB VRAM.

**Inputs:** a start still (required) + an end still (optional → plain I2V), both ideally from 02a.

**VRAM:** ~28 GB resident (bf16 transformer + fp32 VAE). A100-40 uses group offloading, A100-80
runs resident. Below ~40 GB consider 03a's LTX path or the commented 480P/5B variants in §8.

## 1. Config + mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'Yuna'
TRIGGER_TOKEN  = 'sks_vyuna'
NUM_FRAMES     = 81      # 4k+1; 81 = 5 s @ 16 fps
FPS            = 16
MAX_AREA       = 720 * 1280   # 720p target area; aspect follows the start frame
# ─────────────────────────────────────────────────────────────────────────

import os
DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
VID_OUT    = f'{DRIVE_BASE}/outputs/videos/{CHARACTER_NAME}/mode1_wan_flf2v'
STILLS_ROOT = f'{DRIVE_BASE}/outputs/images/{CHARACTER_NAME}'
os.makedirs(VID_OUT, exist_ok=True)
os.environ['HF_HOME'] = f'{DRIVE_BASE}/models/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

print(f'Video out : {VID_OUT}')
print(f'Stills root: {STILLS_ROOT}')

## 2. HuggingFace login + install (uv)
Wan is Apache-2.0 — no gated license.

In [ ]:
import os
hf_token = ''
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
os.environ.setdefault('HF_TOKEN', hf_token)

!pip install -q uv
!uv pip install --system -q --reinstall-package diffusers \
    "diffusers>=0.35.0" transformers accelerate safetensors huggingface_hub ftfy

import torch, diffusers
print('torch', torch.__version__, '| diffusers', diffusers.__version__)
# ── CUDA-torch guard ────────────────────────────────────────────────────────
# uv can silently re-resolve deps and swap Colab's CUDA torch for the PyPI CPU
# wheel ("Torch not compiled with CUDA enabled" at load time). Detect + repair
# here, BEFORE we commit to a multi-GB model download.
import torch as _t
print('torch', _t.__version__, '| cuda', _t.cuda.is_available())
if not _t.cuda.is_available():
    import subprocess
    ver = _t.__version__.split('+')[0]
    print(f'CPU-only torch {ver} detected (uv swapped it). Reinstalling the CUDA build from cu124...')
    subprocess.run(f'uv pip install --system "torch=={ver}" torchvision '
                   '--index-url https://download.pytorch.org/whl/cu124',
                   shell=True, check=True)
    raise RuntimeError(
        'CUDA torch reinstalled on disk. Now: Runtime > Restart runtime, then re-run '
        'this install cell + the model-load cell. (The running kernel still holds the '
        'old CPU torch in memory, so the restart is required — do not skip it.)')


## 3. Load Wan2.1-FLF2V-14B-720P
First run downloads ~30 GB. VAE in fp32 (docs recommend this for decode quality), transformer in
bf16. Log to file, not PIPE.

In [ ]:
import torch, logging
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from transformers import CLIPVisionModel

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {torch.cuda.get_device_name(0)}  ~{vram_gb:.0f} GB')
if vram_gb < 40:
    print('⚠️  FLF2V-14B-720P wants ~40 GB. You will be offloading (slow) — consider 03a (LTX) '
          'or the 480P/5B variants in §8. Continuing anyway.')

LOG = '/content/wan_flf2v_load.log'
logging.basicConfig(filename=LOG, level=logging.INFO)

model_id = "Wan-AI/Wan2.1-FLF2V-14B-720P-diffusers"
image_encoder = CLIPVisionModel.from_pretrained(model_id, subfolder="image_encoder", dtype=torch.float32)
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(model_id, vae=vae, image_encoder=image_encoder,
                                               dtype=torch.bfloat16)

if vram_gb >= 60:
    pipe.to('cuda')
    print('Strategy: resident')
else:
    from diffusers.hooks import apply_group_offloading
    onload, offload = torch.device('cuda'), torch.device('cpu')
    apply_group_offloading(pipe.text_encoder, onload_device=onload, offload_device=offload,
                           offload_type='block_level', num_blocks_per_group=4)
    pipe.transformer.enable_group_offload(onload_device=onload, offload_device=offload,
                                          offload_type='leaf_level', use_stream=True)
    print('Strategy: group offloading')

print('✅ Wan FLF2V loaded.  Log →', LOG)

## 4. Frame-prep helpers + `flf2v()`
Wan's VAE needs dims that are multiples of `vae_scale_factor_spatial * patch_size`. The two helpers
below (`aspect_ratio_resize` then `center_crop_resize`) are straight from the official docs and keep
the start frame's aspect while forcing the end frame onto the same canvas.

In [ ]:
import numpy as np, time
import torchvision.transforms.functional as TF
from pathlib import Path
from PIL import Image
from diffusers.utils import export_to_video, load_image

def aspect_ratio_resize(image, max_area=MAX_AREA):
    aspect_ratio = image.height / image.width
    mod_value = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]
    height = round(np.sqrt(max_area * aspect_ratio)) // mod_value * mod_value
    width  = round(np.sqrt(max_area / aspect_ratio)) // mod_value * mod_value
    return image.resize((width, height)), height, width

def center_crop_resize(image, height, width):
    resize_ratio = max(width / image.width, height / image.height)
    w = round(image.width * resize_ratio); h = round(image.height * resize_ratio)
    return TF.center_crop(image, [w, h]), h, w

def flf2v(start_frame, prompt, end_frame=None, num_frames=NUM_FRAMES, seed=0,
          guidance=5.5, tag=''):
    """
    start_frame : PIL or path (required)
    end_frame   : PIL or path (optional; omit for I2V)
    prompt      : describe MOTION; identity comes from the frames.
    """
    def load(p): return p if isinstance(p, Image.Image) else load_image(p).convert('RGB')
    first = load(start_frame)
    first, h, w = aspect_ratio_resize(first)
    last = None
    if end_frame is not None:
        last = load(end_frame)
        if last.size != first.size:
            last, _, _ = center_crop_resize(last, h, w)

    g = torch.Generator(device='cuda').manual_seed(seed)
    out = pipe(image=first, last_image=last, prompt=prompt,
               height=h, width=w, num_frames=num_frames,
               guidance_scale=guidance, generator=g).frames[0]

    ts = time.strftime('%Y%m%d_%H%M%S')
    out_path = Path(VID_OUT) / (f'{ts}_{tag}.mp4' if tag else f'{ts}.mp4')
    export_to_video(out, str(out_path), fps=FPS)
    print(f'✅ clip → {out_path}  ({num_frames} frames @ {FPS} fps, {w}x{h})')
    return str(out_path)

print('flf2v() ready.')

## 5. Point at keyframes (02a outputs or upload)

In [ ]:
import glob
for d in sorted(glob.glob(f'{STILLS_ROOT}/*'))[-3:]:
    print(d)
    for p in sorted(glob.glob(f'{d}/*.png')):
        print('   ', p)

# Option B: upload fresh keyframes
# from google.colab import files
# files.upload()

START = None   # ← set to a 02a still path
END   = None   # ← set to the target still path, or leave None for I2V
print('Set START (and optionally END) before the run cell.')

## 6. Run a clip

In [ ]:
MOTION_PROMPT = ("She turns her head slowly toward the window, light catching her hair, "
                 "a faint smile. Slow dolly-in, cinematic, shallow depth of field.")

assert START is not None, 'Set START in cell 5 first.'
clip = flf2v(start_frame=START, end_frame=END,
             prompt=f'{TRIGGER_TOKEN}, {MOTION_PROMPT}', seed=0, tag='flf2v')

from IPython.display import Video, display
display(Video(clip, width=720))

## 7. Log to metadata

In [ ]:
import json, os, glob, time
meta_path = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}/metadata.json'
os.makedirs(os.path.dirname(meta_path), exist_ok=True)
meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {'name': CHARACTER_NAME}
clips = sorted(glob.glob(f'{VID_OUT}/*.mp4'))
meta.setdefault('video_log', []).append({
    'ts': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'mode': 'mode1_wan_flf2v',
    'model': 'Wan2.1-FLF2V-14B-720P',
    'clips': clips[-5:],
})
json.dump(meta, open(meta_path, 'w'), indent=2)
print(f'metadata.json updated — {len(clips)} clips in {VID_OUT}')

## 8. Commented alternates (other Wan video models — try later)
The rest of spec 03's video model table, all in diffusers form.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# A) Wan2.1-FLF2V 480P — same first-last-frame task at lower VRAM (~20 GB)
#    Quality step down from 720P, but runs on a 4090/24 GB comfortably.
# ─────────────────────────────────────────────────────────────────────────
# model_id = "Wan-AI/Wan2.1-FLF2V-14B-480P-diffusers"
# image_encoder = CLIPVisionModel.from_pretrained(model_id, subfolder="image_encoder", dtype=torch.float32)
# vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", dtype=torch.float32)
# pipe = WanImageToVideoPipeline.from_pretrained(model_id, vae=vae, image_encoder=image_encoder,
#                                                dtype=torch.bfloat16).to('cuda')
# (then call flf2v() — it uses `pipe`)

# ─────────────────────────────────────────────────────────────────────────
# B) Wan2.2-TI2V-5B — clean I2V (start frame only), 720p@24fps, ~24 GB. Fast on a 4090.
#    Good when you don't have an end frame and just want the best single-start I2V.
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
# mid = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
# vae5 = AutoencoderKLWan.from_pretrained(mid, subfolder="vae", dtype=torch.float32)
# pipe5 = WanImageToVideoPipeline.from_pretrained(mid, vae=vae5, dtype=torch.bfloat16).to('cuda')
# out = pipe5(image=first, prompt=MOTION_PROMPT, height=704, width=1280,
#             num_frames=81, guidance_scale=5.0).frames[0]

# ─────────────────────────────────────────────────────────────────────────
# C) Wan2.1-I2V-14B-720P — strong I2V (start frame only), 720p. ~35 GB.
# ─────────────────────────────────────────────────────────────────────────
# mid = "Wan-AI/Wan2.1-I2V-14B-720P-Diffusers"
# ... same WanImageToVideoPipeline load, drop the last_image arg

# ─────────────────────────────────────────────────────────────────────────
# D) LightX2V 4-step distillation LoRA — cuts Wan sampling from ~50 to ~4 steps.
#    Big speed win for iteration; slight quality trade. Load onto the transformer.
# ─────────────────────────────────────────────────────────────────────────
# pipe.load_lora_weights("lightx2v/Wan2.1-T2V-14B-4steps", adapter_name="fast")
# pipe.set_adapters(["fast"])
# out = pipe(image=first, last_image=last, prompt=MOTION_PROMPT, height=h, width=w,
#            num_frames=81, guidance_scale=1.0, num_inference_steps=4).frames[0]

# ─────────────────────────────────────────────────────────────────────────
# E) Wan CHARACTER video LoRA — for identity retention through motion. Once you've
#    trained a Wan 2.1/2.2 LoRA (musubi-tuner, per spec 01 step 4), load it here so
#    the character holds up better than FLF2V's two-frame conditioning alone.
# ─────────────────────────────────────────────────────────────────────────
# pipe.load_lora_weights("path/or/repo/of/wan-character-lora", adapter_name="yuna")
# pipe.set_adapters(["yuna"])
# and prefix prompts with the LoRA trigger word.

print('Section 8: alternates commented out — enable one at a time.')